#Import Libraries

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install evaluate jiwer -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 13.2 MB/s eta 0:00:00


In [3]:
import torch
import numpy as np
from datasets import load_from_disk
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from dataclasses import dataclass
from typing import Dict, List, Union
import evaluate

#Load Dataset

In [ ]:
# 1. Install dependencies
!pip install -q kagglehub datasets

# 2. Login Kaggle
import kagglehub
kagglehub.login()
# Masukkan token kamu saat diminta

# 3. Download dataset dari Kaggle
import os
from datasets import load_from_disk

dataset_path = kagglehub.dataset_download("nalaprogroup/data-nalapro-project03")
print("Downloaded to:", dataset_path)

# 4. Cek isi folder dulu
print("\nDaftar file:")
for root, dirs, files in os.walk(dataset_path):
    print(root)

# 5. Load dataset
dataset = load_from_disk(dataset_path)
print("✅ Dataset berhasil di-load:", dataset)

# 6. Ekstrak data
def prepare_data(split_data):
    audio_arrays, transcriptions, intent_classes = [], [], []
    for sample in split_data:
        audio_arrays.append(sample['audio']['array'])
        transcriptions.append(sample['transcription'])
        intent_classes.append(sample['intent_class'])
    return audio_arrays, transcriptions, intent_classes

train_audio, train_text, train_labels = prepare_data(dataset['train'])
val_audio,   val_text,   val_labels   = prepare_data(dataset['validation'])
test_audio,  test_text,  test_labels  = prepare_data(dataset['test'])

print(f"\nJumlah train  : {len(train_audio)}")
print(f"Jumlah val    : {len(val_audio)}")
print(f"Jumlah test   : {len(test_audio)}")
print(f"Sample rate   : {dataset['train'][0]['audio']['sampling_rate']} Hz")
print(f"Intent classes: {set(train_labels)}")

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

KeyboardInterrupt: 

In [4]:
print("\n Loading preprocessed dataset...")
save_dir = "/content/drive/MyDrive/data project 3 (Voice Recognition)/minds14_augmented_filter"
dataset = load_from_disk(save_dir)

print(f"\n Dataset loaded!")
print(f"   Train: {len(dataset['train'])} samples (augmented)")
print(f"   Validation: {len(dataset['validation'])} samples")
print(f"   Test: {len(dataset['test'])} samples")


 Loading preprocessed dataset...

 Dataset loaded!
   Train: 1548 samples (augmented)
   Validation: 52 samples
   Test: 51 samples


### Upercase

In [5]:
print("\n Converting text to uppercase...")

def uppercase_text(batch):
    batch["text"] = batch["transcription"].upper()
    return batch

import os
import tempfile

# Create a writable directory for datasets cache if it doesn't exist
# This ensures that temporary files can be written during the map operation
writable_cache_dir = "/tmp/datasets_cache"
os.makedirs(writable_cache_dir, exist_ok=True)

# Generate cache file names for each split in the dataset
# directing them to the writable temporary directory
cache_file_names = {
    split: os.path.join(writable_cache_dir, f"{split}_uppercase_text.arrow")
    for split in dataset.keys()
}

# Map the function, explicitly providing writable cache file paths
dataset = dataset.map(uppercase_text, load_from_cache_file=False, cache_file_names=cache_file_names)

print(f"   Sample: {dataset['train'][0]['text']}")


 Converting text to uppercase...


Map:   0%|          | 0/1548 [00:00<?, ? examples/s]

Map:   0%|          | 0/52 [00:00<?, ? examples/s]

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

   Sample: HELLO I NEED TO CHANGE MY ADDRESS SINCE I MOVED


#Load Model & Processor

In [6]:
print("\n Loading model...")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-base-960h",
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
)


 Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#Preprocess

In [ ]:
'''import re

def clean_text(batch):
  text=batch["text"].lower()
  text=re.sub(r'[^a-z ]','',text)
  batch["text"]=text
  return batch

dataset = dataset.map(clean_text)'''

In [7]:
print("\n Preparing dataset for training...")
max_length = 16000 * 15

def prepare_dataset(batch):
    """Process audio (sudah 16kHz) dan text (sudah uppercase)"""
    audio = batch["audio"]

    # Audio sudah dalam 16kHz dari preprocessing sebelumnya
    inputs = processor(
        audio["array"],
        max_length=max_length,
        sampling_rate=audio["sampling_rate"],  # = 16000
    )
    batch["input_values"] = inputs["input_values"][0]
  # label text
    batch["labels"]=processor(text=batch["text"])["input_ids"]
    return batch


 Preparing dataset for training...


In [8]:
sample = prepare_dataset(dataset["train"][0])
print(sample)

{'audio': {'array': [0.0, -2.5461553377681412e-05, 0.0, 0.00014792315778322518, 0.000244140625, 0.0001463114022044465, 0.0, -2.364019564993214e-05, 0.0, -2.7729505745810457e-05, 0.0, 0.0001508994901087135, 0.000244140625, 0.00014225754421204329, 0.0, -1.791877184587065e-05, 0.0, -3.627346450230107e-05, 0.0, 0.00016509974375367165, 0.000244140625, 0.00011264633212704211, 0.0, 0.00010552084131632, 0.000244140625, 0.00017380370991304517, 0.0, -4.934398748446256e-05, 0.0, 8.087683454505168e-06, 0.0, 3.14392673317343e-05, 0.0, -0.0001497616176493466, -0.000244140625, -0.00014594806998502463, 0.0, 1.8059672584058717e-05, 0.0, 0.00010727663175202906, 0.000244140625, 0.0001965332485269755, 0.0, -9.366575250169262e-05, 0.0, 8.209221414290369e-05, 0.0, -9.976072033168748e-05, 0.0, 0.00021706550614908338, 0.000244140625, -5.032619810663164e-06, -0.000244140625, -0.00020610298088286072, 0.0, 8.578126289648935e-05, 0.0, -6.152484274934977e-05, 0.0, 5.678447996615432e-05, 0.0, -6.957383448025212e-05

In [9]:
encoded_dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=2,
)

Map (num_proc=2):   0%|          | 0/1548 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/52 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/51 [00:00<?, ? examples/s]

In [10]:
sample = encoded_dataset["train"][0]
print("Decode:", processor.decode(sample["labels"]))
print("Original:", dataset["train"][0]["text"])
encoded_dataset["train"]["labels"]

Decode: HELO I NED TO CHANGE MY ADRES SINCE I MOVED
Original: HELLO I NEED TO CHANGE MY ADDRESS SINCE I MOVED


Column([[11, 5, 15, 15, 8, 4, 10, 4, 9, 5, 5, 14, 4, 6, 8, 4, 19, 11, 7, 9, 21, 5, 4, 17, 22, 4, 7, 14, 14, 13, 5, 12, 12, 4, 12, 10, 9, 19, 5, 4, 10, 4, 17, 8, 25, 5, 14], [11, 5, 15, 15, 8, 4, 10, 4, 9, 5, 5, 14, 4, 6, 8, 4, 19, 11, 7, 9, 21, 5, 4, 17, 22, 4, 7, 14, 14, 13, 5, 12, 12, 4, 12, 10, 9, 19, 5, 4, 10, 4, 17, 8, 25, 5, 14], [11, 5, 15, 15, 8, 4, 10, 4, 9, 5, 5, 14, 4, 6, 8, 4, 19, 11, 7, 9, 21, 5, 4, 17, 22, 4, 7, 14, 14, 13, 5, 12, 12, 4, 12, 10, 9, 19, 5, 4, 10, 4, 17, 8, 25, 5, 14], [11, 5, 15, 15, 8, 4, 10, 4, 9, 5, 5, 14, 4, 6, 8, 4, 19, 11, 7, 9, 21, 5, 4, 17, 22, 4, 7, 14, 14, 13, 5, 12, 12, 4, 12, 10, 9, 19, 5, 4, 10, 4, 17, 8, 25, 5, 14], [10, 4, 9, 5, 5, 14, 4, 6, 8, 4, 19, 11, 7, 9, 21, 5, 4, 17, 22, 4, 7, 14, 14, 13, 5, 12, 12]])

# Data Collator

In [11]:
sample = encoded_dataset["train"][0]

print(type(sample["input_values"]))
print(len(sample["input_values"]))
print(type(sample["input_values"][0]))

<class 'list'>
87382
<class 'float'>


In [12]:
print(len(sample["labels"]))

47


In [13]:
for i in range(5):
  sample = encoded_dataset["train"][i]
  print(len(sample["input_values"]), len(sample["labels"]))

87382 47
87382 47
87382 47
87382 47
39594 27


In [14]:
print(len(encoded_dataset["train"][0]["input_values"])/16000)

5.461375


In [15]:
print(sample["input_values"]["path"])

TypeError: list indices must be integers or slices, not str

In [16]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorCTCWithPadding:
  processor: Any
  tokenizer: Any

  def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
    input_features = [{"input_values": feature["input_values"]} for feature in features]
    label_features = [{"input_ids": feature["labels"]} for feature in features]

    batch = self.processor.pad(
        input_features,
        padding=True,
        return_tensors="pt",
    )

    labels_batch = self.tokenizer.pad(
        label_features,
        padding=True,
        return_tensors="pt",
    )

    labels = labels_batch["input_ids"].masked_fill(
        labels_batch.attention_mask.ne(1), -100
    )

    batch["labels"] = labels

    return batch

In [17]:
data_collator = DataCollatorCTCWithPadding(processor=processor, tokenizer=processor.tokenizer)

In [ ]:
'''def simple_collator(features):
    input_values = [f["input_values"] for f in features]
    labels = [f["labels"] for f in features]

    batch = processor.pad(
        {"input_values": input_values},
        padding=True,
        return_tensors="pt"
    )

    labels_batch = processor.pad(
        labels={"input_ids": labels},
        padding=True,
        return_tensors="pt"
    )

    labels = labels_batch["input_ids"].masked_fill(
        labels_batch.attention_mask.ne(1), -100
    )

    batch["labels"] = labels
    return batch

data_collator = simple_collator

In [18]:
batch = data_collator([encoded_dataset["train"][0]])
print(batch.keys())

KeysView({'input_values': tensor([[-0.0074, -0.0094, -0.0074,  ..., -0.0199, -0.0074, -0.0012]]), 'labels': tensor([[11,  5, 15, 15,  8,  4, 10,  4,  9,  5,  5, 14,  4,  6,  8,  4, 19, 11,
          7,  9, 21,  5,  4, 17, 22,  4,  7, 14, 14, 13,  5, 12, 12,  4, 12, 10,
          9, 19,  5,  4, 10,  4, 17,  8, 25,  5, 14]])})


#Metrics

In [19]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

    print("PRED:", pred_str[:2])
    print("LABEL:", label_str[:2])

In [20]:
print(encoded_dataset["train"][0])

{'input_values': [-0.007401576265692711, -0.009372688829898834, -0.007401576265692711, 0.004049931652843952, 0.011498630978167057, 0.003925156779587269, -0.007401576265692711, -0.0092316884547472, -0.007401576265692711, -0.009548263624310493, -0.007401576265692711, 0.004280345048755407, 0.011498630978167057, 0.0036113266833126545, -0.007401576265692711, -0.008788762614130974, -0.007401576265692711, -0.010209695436060429, -0.007401576265692711, 0.005379660986363888, 0.011498630978167057, 0.0013189672026783228, -0.007401576265692711, 0.0007673455984331667, 0.011498630978167057, 0.006053480785340071, -0.007401576265692711, -0.01122155413031578, -0.007401576265692711, -0.00677546625956893, -0.007401576265692711, -0.004967697896063328, -0.007401576265692711, -0.018995407968759537, -0.026301782578229904, -0.01870018243789673, -0.007401576265692711, -0.0060034822672605515, -0.007401576265692711, 0.0009032705565914512, 0.011498630978167057, 0.00781309325248003, -0.007401576265692711, -0.014652

#Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./wav2vec2-minds14-base-960h",
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=50,
    save_steps=50,
    learning_rate=3e-5,
    per_device_train_batch_size=4,  # Reduced from 8 to 4
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    warmup_steps=0.1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=5),
    ]
)

print("\n Starting training...")
trainer.train()


 Starting training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss


#Save Model

In [ ]:
print("\n Saving final model...")

model_save_path = "./wav2vec2-minds14-final"
trainer.save_model(model_save_path)
processor.save_pretrained(model_save_path)

print(f" Model saved to {model_save_path}")